# RAG-Based Customer Support Assistant
Using LangGraph + Human-in-the-Loop (HITL) + Groq LLM



###  System Architecture

```
PDF Documents
     ↓
Document Loader  (PyPDFLoader)
     ↓
Chunking  (RecursiveCharacterTextSplitter)
     ↓
Embeddings  (HuggingFace sentence-transformers)
     ↓
ChromaDB  (persistent vector store)
     ↓
User Query
     ↓
Intent Detection  (LangGraph node)
     ↓
LangGraph Workflow Engine
     ↓
Conditional Routing
    ├─── FAQ / General  ──→  Retriever  ──→  Groq LLM  ──→  Answer
    └─── Low Confidence / Complaint / Sensitive  ──→  HITL Escalation
```


##  Section 1 — Install Dependencies

> Run this cell first. Restart runtime after installation if prompted.

In [ ]:
!pip install -q langchain langchain-community langchain-groq langgraph \
               chromadb sentence-transformers pypdf reportlab \
               groq typing-extensions

print(" All packages installed.")

##  Section 2 — Configuration & Imports

All tunable parameters live here. Change chunk size, top-k, thresholds,
and the Groq API key in one place without touching downstream code.


In [ ]:
import os, json, uuid, time, logging
from dataclasses import dataclass, field
from typing import List, Optional, Literal

import numpy as np
from typing_extensions import TypedDict

# LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Embeddings — HuggingFace sentence-transformers
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector store
import chromadb

# LangGraph
from langgraph.graph import StateGraph, END

# Groq LLM
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

logging.basicConfig(level=logging.WARNING)

# ── API KEY ──
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "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")

# ── TUNABLE PARAMETERS ─────────────────────────────────────────────────────
GROQ_MODEL         = "llama3-8b-8192"
EMBED_MODEL        = "all-MiniLM-L6-v2"
COLLECTION_NAME    = "support_kb"        # ChromaDB collection name
CHROMA_DIR         = "/tmp/chroma_db"

CHUNK_SIZE         = 512
CHUNK_OVERLAP      = 64
TOP_K              = 4

# Routing thresholds
CONFIDENCE_THRESH  = 0.55
ESCALATION_KEYWORDS = {
    "refund", "complaint", "legal", "lawsuit", "fraud",
    "urgent", "compensation", "attorney", "sue", "angry",
    "unacceptable", "damage", "broken", "scam"
}

print(" Configuration loaded.")
print(f"   Model    : {GROQ_MODEL}")
print(f"   Embedder : {EMBED_MODEL}")
print(f"   Chunk    : {CHUNK_SIZE} tokens  |  Overlap: {CHUNK_OVERLAP}")
print(f"   Top-K    : {TOP_K}  |  Conf threshold: {CONFIDENCE_THRESH}")


##  Section 3 — Knowledge Base PDF

We programmatically create a realistic customer-support knowledge-base PDF.
In a real deployment you would upload your own product manuals, FAQs, or policy documents.



In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

PDF_PATH = "/tmp/support_knowledge_base.pdf"

def create_sample_pdf(path: str) -> None:
    doc = SimpleDocTemplate(path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    kb_content = [
        ("Customer Support Knowledge Base", "Title"),
        ("1. Return & Refund Policy", "Heading1"),
        ("Customers may return any product within 30 days of purchase for a full refund. "
         "The item must be in its original packaging and unused condition. "
         "To initiate a return, contact support@company.com with your order number. "
         "Refunds are processed within 5-7 business days to the original payment method. "
         "After 30 days, returns are accepted only for defective items covered under warranty.", "Normal"),

        ("2. Warranty Policy", "Heading1"),
        ("All products carry a 12-month limited warranty from the date of purchase. "
         "The warranty covers manufacturing defects and hardware failures under normal use. "
         "It does NOT cover physical damage, water damage, or misuse. "
         "To make a warranty claim, provide proof of purchase and a description of the defect. "
         "Approved warranty claims result in a free replacement or repair within 10 business days.", "Normal"),

        ("3. Shipping & Delivery", "Heading1"),
        ("Standard shipping takes 5-7 business days within the country. "
         "Express shipping (2-3 days) is available for an additional fee. "
         "International orders may take 10-20 business days depending on customs clearance. "
         "Orders above $50 qualify for free standard shipping. "
         "You will receive a tracking number via email once your order ships.", "Normal"),

        ("4. Account & Password Management", "Heading1"),
        ("To reset your password, visit the login page and click 'Forgot Password'. "
         "An email link valid for 24 hours will be sent to your registered address. "
         "For account deletion, send a request to privacy@company.com. "
         "Two-factor authentication (2FA) can be enabled from Account Settings > Security. "
         "If you are locked out after 5 failed attempts, wait 30 minutes or contact support.", "Normal"),

        ("5. Product Troubleshooting", "Heading1"),
        ("If your device does not power on, try holding the power button for 10 seconds. "
         "Ensure the battery is charged for at least 30 minutes before the first use. "
         "Factory reset can be performed via Settings > System > Reset. "
         "Software updates are available in Settings > About > Check for Updates. "
         "For connectivity issues, toggle airplane mode on and off, then reconnect.", "Normal"),

        ("6. Payment & Billing", "Heading1"),
        ("We accept Visa, Mastercard, American Express, PayPal, and bank transfers. "
         "All transactions are secured with 256-bit SSL encryption. "
         "Invoices are emailed automatically after each purchase. "
         "For billing disputes, contact billing@company.com within 60 days of the charge. "
         "Subscription plans can be cancelled anytime; access continues until the period ends.", "Normal"),

        ("7. Privacy & Data", "Heading1"),
        ("We do not sell customer data to third parties. "
         "Data is stored on encrypted servers compliant with GDPR and CCPA. "
         "Customers may request a data export or deletion at any time. "
         "Cookies are used only for session management and analytics with consent. "
         "Review our full Privacy Policy at company.com/privacy.", "Normal"),
    ]

    for text, style in kb_content:
        story.append(Paragraph(text, styles[style]))
        story.append(Spacer(1, 8))

    doc.build(story)
    print(f" Sample knowledge base PDF created: {path}")

create_sample_pdf(PDF_PATH)


##  Section 4 — Document Ingestion Pipeline

Raw PDFs must be converted to structured text before any AI processing.
`PyPDFLoader` extracts text page-by-page and preserves metadata (filename, page number)
which becomes critical for source citation later.


In [ ]:
def load_documents(pdf_path: str) -> List[Document]:
    """
    Load a PDF and return a list of LangChain Document objects.

    Args:
        pdf_path: Path to the PDF file

    Returns:
        List of Document objects, one per page, each with:
          - page_content: extracted text
          - metadata: {'source': filename, 'page': page_number}

    Raises:
        FileNotFoundError: if the PDF does not exist
    """
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"❌ PDF not found: {pdf_path}")

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # Enrich metadata
    for doc in documents:
        doc.metadata["filename"] = os.path.basename(pdf_path)

    print(f"✅ Loaded {len(documents)} pages from '{os.path.basename(pdf_path)}'")
    for i, doc in enumerate(documents):
        preview = doc.page_content[:80].replace("\n", " ")
        print(f"   Page {doc.metadata.get('page', i)}: {preview}...")

    return documents

# ── Run the loader ───────────────────────────────────────────────────────────
raw_documents = load_documents(PDF_PATH)


##  Section 5 — Chunking Module


- LLMs have limited context windows (~4k–32k tokens)
- Embedding entire pages reduces retrieval precision
- Smaller chunks → each chunk is more topically focused → better similarity matching
- Overlap prevents loss of information at chunk boundaries (a sentence split across two chunks)

`RecursiveCharacterTextSplitter` tries separators in order: `\n\n → \n → . → space`,
always preferring semantic breaks over arbitrary character cuts.


In [ ]:
def chunk_documents(documents: List[Document],
                    chunk_size: int = CHUNK_SIZE,
                    chunk_overlap: int = CHUNK_OVERLAP) -> List[Document]:
    """
    Split documents into overlapping chunks for embedding.

    The RecursiveCharacterTextSplitter tries these separators in order:
      '\\n\\n'  →  '\\n'  →  '.'  →  ' '
    This ensures splits happen at natural semantic boundaries.

    Args:
        documents  : List of raw Document objects from the loader
        chunk_size : Max character length per chunk (default: 512)
        chunk_overlap: Overlap between adjacent chunks (default: 64)

    Returns:
        List of chunked Document objects with enriched metadata
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""],
        length_function=len,
    )

    chunks = splitter.split_documents(documents)

    # Assign a unique ID to each chunk for ChromaDB upsert
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"]    = str(uuid.uuid4())
        chunk.metadata["chunk_index"] = i

    print(f" Chunking complete: {len(documents)} pages → {len(chunks)} chunks")
    print(f"   Chunk size: {chunk_size}  |  Overlap: {chunk_overlap}")

    if chunks:
        sample = chunks[min(2, len(chunks)-1)]
        print(f"\n   Sample chunk #{sample.metadata['chunk_index']}:")
        print(f"   Source : {sample.metadata.get('source','?')} | Page: {sample.metadata.get('page','?')}")
        print(f"   Length : {len(sample.page_content)} chars")
        print(f"   Text   : {sample.page_content[:200]}...")

    return chunks

# ── Run chunker ───
chunks = chunk_documents(raw_documents)


##  Section 6 — Embedding System + ChromaDB Storage

**WHY embeddings are used**:
- Text cannot be compared mathematically. Embeddings convert text into dense numerical vectors.
- Semantically similar sentences produce similar vectors (high cosine similarity).
- This allows us to find *meaningfully relevant* chunks even if the query uses different words.

**WHY ChromaDB**:
- Zero-config persistent vector database
- Built-in cosine similarity search
- Native LangChain integration
- Can scale to millions of vectors with the HTTP server mode


In [ ]:
print(" Loading HuggingFace embedding model (first time downloads ~90 MB)...")
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}  # normalise → cosine via dot product
)
print(f" Embedding model '{EMBED_MODEL}' loaded.")

# ── ChromaDB client
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Cosine similarity space ensures scores are in [0, 1]
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
print(f" ChromaDB collection '{COLLECTION_NAME}' ready at {CHROMA_DIR}")

# ── Upsert function ───
def upsert_chunks_to_chroma(chunks: List[Document]) -> None:
    """
    Embed all chunks and store them in ChromaDB.

    Steps:
      1. Extract text from each chunk
      2. Batch-encode with HuggingFace embeddings
      3. Upsert (insert or update) into ChromaDB with metadata

    Uses upsert (not add) so re-running this cell won't duplicate data.
    """
    texts     = [c.page_content               for c in chunks]
    ids       = [c.metadata["chunk_id"]       for c in chunks]
    metadatas = [{
        "source"      : str(c.metadata.get("source", "unknown")),
        "page"        : str(c.metadata.get("page", 0)),
        "chunk_index" : str(c.metadata.get("chunk_index", 0)),
        "filename"    : str(c.metadata.get("filename", "unknown")),
    } for c in chunks]

    print(f" Embedding {len(texts)} chunks (this may take 10-30 seconds)...")
    embeddings = embedding_model.embed_documents(texts)

    collection.upsert(
        ids        = ids,
        embeddings = embeddings,
        documents  = texts,
        metadatas  = metadatas
    )
    print(f"✅ {len(chunks)} chunks stored in ChromaDB.")
    print(f"   Total vectors in collection: {collection.count()}")

upsert_chunks_to_chroma(chunks)


##  Section 7 — Retrieval Layer

The retrieval layer takes a user query, embeds it with the **same** model used during ingestion,
and finds the top-k most semantically similar chunks in ChromaDB.

The cosine **distance** returned by ChromaDB is converted to a **similarity score** (1 - distance)
so higher = better. The mean similarity across retrieved chunks becomes the system's **confidence score**,
which drives the routing decision downstream.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class RetrievalResult:
    """Structured output from the retrieval layer."""
    chunks     : List[str]
    scores     : List[float]
    sources    : List[str]
    confidence : float

def retrieve(query: str, top_k: int = TOP_K) -> RetrievalResult:
    """
    Embed the query and retrieve the top-k most relevant chunks.

    Args:
        query : the user's natural-language question
        top_k : number of chunks to return (default: TOP_K from config)

    Returns:
        RetrievalResult with chunks, similarity scores, sources, and confidence
    """
    n_available = collection.count()
    if n_available == 0:
        print("⚠️  ChromaDB collection is empty — run ingestion first.")
        return RetrievalResult([], [], [], 0.0)

    k = min(top_k, n_available)

    # Embed the query using the SAME model used at ingestion time
    query_embedding = embedding_model.embed_query(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    docs    = results["documents"][0] if results["documents"] else []
    dists   = results["distances"][0]  if results["distances"]  else []
    metas   = results["metadatas"][0]  if results["metadatas"]  else []


    scores  = [round(max(0.0, 1.0 - d), 4) for d in dists]
    sources = [f"{m.get('filename','?')} | page {m.get('page','?')}" for m in metas]
    confidence = round(float(np.mean(scores)), 4) if scores else 0.0

    return RetrievalResult(chunks=docs, scores=scores,
                           sources=sources, confidence=confidence)

# retrieval test
print("🔍 Retrieval test: 'What is the refund policy?'")
test_result = retrieve("What is the refund policy?")
print(f"   Confidence  : {test_result.confidence}")
print(f"   Top chunk   : {test_result.chunks[0][:150] if test_result.chunks else 'None'}...")
print(f"   Source      : {test_result.sources[0] if test_result.sources else 'N/A'}")


##  Section 8 — Groq LLM Layer


The prompt is carefully structured to:
1. Force the LLM to answer **only** from retrieved context (prevents hallucination)
2. Require source citation in the answer
3. Acknowledge when information is not available (graceful uncertainty)


In [ ]:
llm = ChatGroq(
    groq_api_key = GROQ_API_KEY,
    model_name   = GROQ_MODEL,
    temperature  = 0.1,     # low temperature → factual, consistent answers
    max_tokens   = 512,
)

def build_rag_prompt(query: str,
                     chunks: List[str],
                     sources: List[str],
                     human_note: Optional[str] = None) -> str:
    """
    Build a structured prompt that grounds the LLM in retrieved context.

    Design principles:
      1. System instruction prevents hallucination
      2. Each context block is labelled with its source
      3. Optional human_note integrates HITL agent guidance
      4. Question is asked last (recency bias helps the model focus)
    """
    context_blocks = "\n\n".join(
        f"[Source {i+1}: {src}]\n{chunk}"
        for i, (chunk, src) in enumerate(zip(chunks, sources))
    )

    human_section = ""
    if human_note:
        human_section = f"""

--- AGENT GUIDANCE (from human support specialist) ---
{human_note}
------------------------------------------------------"""

    return f"""You are a professional customer support assistant.
Your job is to answer customer questions accurately and helpfully.

STRICT RULES:
1. Answer ONLY using the context provided below. Do NOT use prior knowledge.
2. If the answer is not in the context, say: "I don't have that information in my knowledge base. Please contact support@company.com"
3. Always mention which source your answer comes from.
4. Be concise, polite, and professional.
5. If the question involves refunds, legal matters, or complaints — acknowledge empathetically.
{human_section}

--- KNOWLEDGE BASE CONTEXT ---
{context_blocks}
------------------------------

Customer Question: {query}

Answer:"""

def call_groq(prompt: str) -> str:
    """Call the Groq LLM and return the text response."""
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content.strip()
    except Exception as e:
        # Fallback — never crash the pipeline due to LLM errors
        print(f"⚠️  Groq API error: {e}")
        return ("I'm unable to process your request right now. "
                "Please contact support@company.com or try again later.")

print(" Groq LLM initialised.")
print(f"   Model      : {GROQ_MODEL}")
print(f"   Temperature: 0.1 (factual mode)")

# ── Note for users without a Groq key ────────────────────────────────────────
if GROQ_API_KEY == "YOUR_GROQ_API_KEY_HERE":
    print("\n⚠️  GROQ_API_KEY not set. LLM calls will return a placeholder.")
    print("   Get a free key at: https://console.groq.com")
    # Override with a mock for demo purposes
    def call_groq(prompt: str) -> str:
        lines = prompt.split("\n")
        for line in lines:
            if line.startswith("Customer Question:"):
                q = line.replace("Customer Question:", "").strip()
                return (f"[MOCK — set GROQ_API_KEY for real answers] "
                        f"Based on the knowledge base, here is the answer to: '{q}'. "
                        f"Please refer to the retrieved context above for details.")
        return "[MOCK RESPONSE] Please set your GROQ_API_KEY."
    print("   Mock LLM activated for demonstration.")


##  Section 9 — LangGraph Workflow Engine



In [ ]:
class GraphState(TypedDict):
    query          : str
    intent         : str
    scores         : List[float]
    sources        : List[str]
    response       : str
    confidence     : float
    escalation     : bool
    human_response : Optional[str]
    error          : Optional[str]


# NODE 1 — classify_intent
def node_classify_intent(state: GraphState) -> GraphState:
    """
    Node 1: Intent Classification

    Purpose: Understand WHAT type of question this is before retrieval.
    This shapes both the retrieval strategy and the routing decision.

    Intent categories:
      - 'complaint'  : customer is expressing dissatisfaction or requesting compensation
      - 'sensitive'  : legal, fraud, privacy-related queries
      - 'technical'  : troubleshooting, setup, account issues
      - 'faq'        : policy questions, general how-to
      - 'other'      : unclassified
    """
    query = state["query"].lower()

    # Rule-based intent classification

    if any(kw in query for kw in ["complaint", "compensation", "angry",
                                   "unacceptable", "damage", "broken", "bad"]):
        intent = "complaint"
    elif any(kw in query for kw in ["legal", "lawsuit", "attorney", "sue",
                                     "fraud", "scam", "privacy"]):
        intent = "sensitive"
    elif any(kw in query for kw in ["password", "login", "reset", "account",
                                     "install", "update", "troubleshoot", "error"]):
        intent = "technical"
    elif any(kw in query for kw in ["refund", "return", "warranty", "shipping",
                                     "delivery", "policy", "cancel"]):
        intent = "faq"
    else:
        intent = "other"

    print(f"  [Node 1 — classify_intent] intent='{intent}'")
    return {**state, "intent": intent}


# NODE 2 — retrieve_context  (Retrieval Node)

def node_retrieve_context(state: GraphState) -> GraphState:
    """
    Node 2: Retrieval from ChromaDB

    Uses the query to perform semantic similarity search.
    Populates retrieved_docs, scores, sources, and confidence in state.
    """
    result = retrieve(state["query"], top_k=TOP_K)

    print(f"  [Node 2 — retrieve_context] confidence={result.confidence:.3f} | chunks={len(result.chunks)}")

    return {
        **state,
        "retrieved_docs" : result.chunks,
        "scores"         : result.scores,
        "sources"        : result.sources,
        "confidence"     : result.confidence,
    }


# NODE 3 — generate_answer  (Response Generation Node)

def node_generate_answer(state: GraphState) -> GraphState:
    """
    Node 3: LLM Answer Generation

    Builds a structured RAG prompt from retrieved context and calls Groq.
    The answer is grounded — the LLM cannot hallucinate beyond the context.
    """
    print(f"  [Node 3 — generate_answer] calling Groq LLM...")

    prompt = build_rag_prompt(
        query   = state["query"],
        chunks  = state["retrieved_docs"],
        sources = state["sources"]
    )
    answer = call_groq(prompt)

    print(f"  [Node 3 — generate_answer] answer generated ({len(answer)} chars)")
    return {**state, "response": answer, "escalation": False}


# NODE 4 — hitl_escalation  (HITL Escalation Node)

def node_hitl_escalation(state: GraphState) -> GraphState:
    """
    Node 4: Human-in-the-Loop Escalation

    Triggered when:
      a) Confidence is below threshold (retrieval didn't find relevant context)
      b) Query intent is 'complaint' or 'sensitive'
      c) Query contains escalation keywords

    What this node does:
      1. Logs the escalation with full context
      2. Simulates human agent review (in production: sends to ticket system)
      3. Injects a human response back into the graph state
      4. Generates a final answer using both retrieved context + human guidance
    """
    reason = "unknown"
    if state["confidence"] < CONFIDENCE_THRESH:
        reason = f"low confidence ({state['confidence']:.2f} < {CONFIDENCE_THRESH})"
    elif state["intent"] in ("complaint", "sensitive"):
        reason = f"sensitive intent: '{state['intent']}'"
    else:
        reason = "escalation keyword detected"

    print(f"  [Node 4 — hitl_escalation] 🚨 ESCALATED — reason: {reason}")

    escalation_log = {
        "timestamp"  : time.strftime("%Y-%m-%dT%H:%M:%S"),
        "query"      : state["query"],
        "intent"     : state["intent"],
        "confidence" : state["confidence"],
        "reason"     : reason,
        "top_chunks" : state["retrieved_docs"][:2],
    }
    print(f"  [HITL Log] {json.dumps(escalation_log, indent=4)}")

    # ── Simulate human agent response
    simulated_responses = {
        "complaint" : (
            "This customer is reporting a product issue. "
            "Acknowledge the inconvenience, apologise sincerely, "
            "and offer to process a full refund or replacement as per our policy. "
            "Escalate to Tier-2 support if the damage is significant."
        ),
        "sensitive" : (
            "This query involves potential legal/privacy concerns. "
            "Do NOT make any commitments. Direct the customer to legal@company.com "
            "and our official Privacy Policy at company.com/privacy."
        ),
        "other" : (
            "The knowledge base may not cover this query. "
            "Provide the best available answer from context "
            "and offer to connect the customer with a specialist."
        ),
    }
    human_note = simulated_responses.get(
        state["intent"],
        simulated_responses["other"]
    )
    print(f"  [HITL Agent] Simulated response injected.")

    # Generate augmented answer with human guidance
    prompt = build_rag_prompt(
        query      = state["query"],
        chunks     = state["retrieved_docs"],
        sources    = state["sources"],
        human_note = human_note
    )
    answer = call_groq(prompt)

    return {
        **state,
        "response"       : f"[ Agent-Assisted] {answer}",
        "escalation"     : True,
        "human_response" : human_note,
    }


# NODE 5 — format_output  (Output Node)

def node_format_output(state: GraphState) -> GraphState:
    """
    Node 5: Output Formatting

    Adds metadata (sources, confidence, escalation flag) to the response.
    In production: this node could format for JSON API, HTML, or SMS.
    """
    citation_text = ""
    if state["sources"]:
        unique_sources = list(dict.fromkeys(state["sources"]))  # deduplicate
        citation_text  = "\n\n📄 Sources: " + " | ".join(unique_sources)

    formatted = state["response"] + citation_text

    print(f"  [Node 5 — format_output] Output ready. escalated={state['escalation']}")
    return {**state, "response": formatted}


# CONDITIONAL EDGE — route_decision

def route_decision(state: GraphState) -> Literal["generate", "escalate"]:
    """
    Routing Node (conditional edge function)

    Evaluates multiple signals to decide the execution path:

    ESCALATE if any of:
      1. No chunks retrieved (confidence == 0.0)
      2. Mean cosine similarity < CONFIDENCE_THRESH (0.55)
      3. Intent is 'complaint' or 'sensitive'
      4. Query contains an escalation keyword

    GENERATE if:
      - Sufficient relevant context found
      - Non-sensitive intent
      - No escalation keywords

    This is the heart of the decision-making architecture.
    """
    query_lower = state["query"].lower()

    # Signal 1: empty retrieval
    if not state["retrieved_docs"]:
        print(f"  [Router] → ESCALATE (no chunks found)")
        return "escalate"

    # Signal 2: low retrieval confidence
    if state["confidence"] < CONFIDENCE_THRESH:
        print(f"  [Router] → ESCALATE (confidence {state['confidence']:.2f} < {CONFIDENCE_THRESH})")
        return "escalate"

    # Signal 3: complaint or sensitive intent
    if state["intent"] in ("complaint", "sensitive"):
        print(f"  [Router] → ESCALATE (intent: {state['intent']})")
        return "escalate"

    # Signal 4: hard escalation keywords
    if any(kw in query_lower for kw in ESCALATION_KEYWORDS):
        print(f"  [Router] → ESCALATE (keyword match)")
        return "escalate"

    print(f"  [Router] → GENERATE (confidence {state['confidence']:.2f} ✅)")
    return "generate"


# BUILD THE LANGGRAPH

def build_graph() -> object:
    """
    Construct and compile the LangGraph state machine.

    Graph topology:
      START → classify_intent → retrieve_context → [conditional] → ...
        if generate: generate_answer → format_output → END
        if escalate: hitl_escalation → format_output → END
    """
    graph = StateGraph(GraphState)

    # Register all nodes
    graph.add_node("classify_intent",  node_classify_intent)
    graph.add_node("retrieve_context", node_retrieve_context)
    graph.add_node("generate_answer",  node_generate_answer)
    graph.add_node("hitl_escalation",  node_hitl_escalation)
    graph.add_node("format_output",    node_format_output)

    graph.set_entry_point("classify_intent")

    graph.add_edge("classify_intent", "retrieve_context")

    graph.add_conditional_edges(
        "retrieve_context",
        route_decision,
        {
            "generate" : "generate_answer",
            "escalate" : "hitl_escalation",
        }
    )

    graph.add_edge("generate_answer", "format_output")
    graph.add_edge("hitl_escalation", "format_output")
    graph.add_edge("format_output",   END)

    compiled = graph.compile()
    print(" LangGraph compiled successfully.")
    print("   Nodes: classify_intent → retrieve_context → [route] → generate_answer | hitl_escalation → format_output")
    return compiled

support_graph = build_graph()


##  Section 10 — Full Pipeline Runner

A clean `run_query()` function wraps the entire LangGraph execution,
handles errors gracefully, and returns a structured `AnswerResponse`.


In [ ]:
@dataclass
class AnswerResponse:
    """Structured output from one full query execution."""
    query      : str
    intent     : str
    answer     : str
    sources    : List[str]
    confidence : float
    escalated  : bool
    latency_ms : int

    def display(self):
        """Pretty-print the response in a notebook-friendly format."""
        sep  = "═" * 70
        mark = " HITL" if self.escalated else "🤖 AI"
        print(f"\n{sep}")
        print(f"  {mark}  |  Intent: {self.intent}  |  Confidence: {self.confidence:.2f}  |  {self.latency_ms}ms")
        print(sep)
        print(f"  Q: {self.query}")
        print(f"  A: {self.answer}")
        print(sep)


def run_query(user_query: str) -> AnswerResponse:
    """
    Execute the full RAG pipeline for a given user query.

    Args:
        user_query: natural-language customer question

    Returns:
        AnswerResponse with answer, sources, confidence, escalation flag, latency
    """
    print(f"\n{'─'*60}")
    print(f"🔹 Query: {user_query}")
    print(f"{'─'*60}")

    t0 = time.time()

    initial_state: GraphState = {
        "query"          : user_query,
        "intent"         : "",
        "retrieved_docs" : [],
        "scores"         : [],
        "sources"        : [],
        "response"       : "",
        "confidence"     : 0.0,
        "escalation"     : False,
        "human_response" : None,
        "error"          : None,
    }

    try:
        final_state = support_graph.invoke(initial_state)
    except Exception as e:
        print(f" Pipeline error: {e}")
        final_state = {
            **initial_state,
            "response"  : "An internal error occurred. Please contact support@company.com.",
            "error"     : str(e),
        }

    latency = int((time.time() - t0) * 1000)

    return AnswerResponse(
        query      = user_query,
        intent     = final_state.get("intent", "unknown"),
        answer     = final_state.get("response", ""),
        sources    = final_state.get("sources",  []),
        confidence = final_state.get("confidence", 0.0),
        escalated  = final_state.get("escalation", False),
        latency_ms = latency,
    )

print("✅ Pipeline runner ready.")


##  Section 11 — Test Queries with Routing Demo



In [41]:

test_queries = [
    "What is the refund policy?",
    "Can I return a product after 10 days?",
    "How do I reset my password?",
    "What shipping options are available?",

    "I want compensation for product damage, this is unacceptable.",
    "I am considering legal action due to fraud on my account.",
]

results = []
for q in test_queries:
    resp = run_query(q)
    resp.display()
    results.append(resp)



────────────────────────────────────────────────────────────
🔹 Query: What is the refund policy?
────────────────────────────────────────────────────────────
  [Node 1 — classify_intent] intent='faq'
  [Node 2 — retrieve_context] confidence=0.498 | chunks=4
 Pipeline error: 'retrieved_docs'

══════════════════════════════════════════════════════════════════════
  🤖 AI  |  Intent:   |  Confidence: 0.00  |  43ms
══════════════════════════════════════════════════════════════════════
  Q: What is the refund policy?
  A: An internal error occurred. Please contact support@company.com.
══════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
🔹 Query: Can I return a product after 10 days?
────────────────────────────────────────────────────────────
  [Node 1 — classify_intent] intent='faq'
  [Node 2 — retrieve_context] confidence=0.540 | chunks=4
 Pipeline error: 'retrieved_docs'

═════════════════════════════════════

##  Section 12 — Results Summary

In [ ]:

print("\n" + "═"*90)
print(f"  {'QUERY':<45} {'INTENT':<12} {'CONF':>6} {'ESCALATED':<12} {'MS':>6}")
print("═"*90)
for r in results:
    q_short = r.query[:44]
    esc_sym = "✅ YES 🤝" if r.escalated else "❌ No"
    print(f"  {q_short:<45} {r.intent:<12} {r.confidence:>6.2f} {esc_sym:<12} {r.latency_ms:>6}")
print("═"*90)
print(f"  Total queries: {len(results)} | Escalated: {sum(r.escalated for r in results)} | Direct: {sum(not r.escalated for r in results)}")


##  Section 13 — Interactive Chat Loop

Run this cell to enter an interactive Q&A session with the system.
Type `exit` to quit.


In [42]:
print(" RAG Customer Support Assistant — Interactive Mode")
print("   Type your question and press Enter. Type 'exit' to quit.\n")

while True:
    try:
        user_input = input("You: ").strip()
    except EOFError:
        # Non-interactive environment (Colab auto-run)
        print("(Skipping interactive loop in auto-run mode)")
        break

    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit", "q"):
        print("Goodbye! 👋")
        break

    response = run_query(user_input)
    response.display()


 RAG Customer Support Assistant — Interactive Mode
   Type your question and press Enter. Type 'exit' to quit.

You: Can I return a product after 5 days?

────────────────────────────────────────────────────────────
🔹 Query: Can I return a product after 5 days?
────────────────────────────────────────────────────────────
  [Node 1 — classify_intent] intent='faq'
  [Node 2 — retrieve_context] confidence=0.541 | chunks=4
 Pipeline error: 'retrieved_docs'

══════════════════════════════════════════════════════════════════════
  🤖 AI  |  Intent:   |  Confidence: 0.00  |  31ms
══════════════════════════════════════════════════════════════════════
  Q: Can I return a product after 5 days?
  A: An internal error occurred. Please contact support@company.com.
══════════════════════════════════════════════════════════════════════
You: How long does delivery take?

────────────────────────────────────────────────────────────
🔹 Query: How long does delivery take?
──────────────────────────────────